In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Memuat variabel lingkungan dari file .env
load_dotenv()

# Menentukan lokasi direktori proyek dan aset
pathRes = Path('../res')
pathInputs = pathRes / 'inputs'
pathOutputs = pathRes / 'outputs'

# Menentukan direktori cache Hugging Face
pathHf = Path('D:/ai/hf')

# Membuat direktori output jika belum tersedia
pathOutputs.mkdir(parents=True, exist_ok=True)

# Mengatur lokasi penyimpanan model dan cache Hugging Face
os.environ['HF_HOME'] = str(pathHf)

# Menentukan perangkat untuk proses inferensi
device = 'cuda'

In [2]:
# Mengukur waktu eksekusi.
import time

# PyTorch untuk komputasi tensor dan inferensi GPU
import torch 

# Pillow untuk membaca dan memproses gambar
from PIL import Image, ImageOps 

# Memuat model transformer dan pipeline LTX
from diffusers import AutoModel, LTXConditionPipeline

# Mengatur gambar sebagai kondisi awal video.
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition

# Mengaktifkan mekanisme pemindahan model antara CPU dan GPU.
from diffusers.hooks import apply_group_offloading

# Fungsi untuk mengekspor frame menjadi video
from diffusers.utils import export_to_video

In [3]:
# ID repositori model di Hugging Face
REPO_ID = 'Lightricks/LTX-Video-0.9.8-13B-distilled'


# Sampling
# Menentukan timestep yang digunakan selama proses denoising
TIMESTEPS = [
  1000, 993, 987, 981,
  975, 909, 725, 0.03,
]

# Mengatur kekuatan classifier-free guidance
GUIDANCE_SCALE = 1.0

# Mengatur rescaling pada hasil classifier-free guidance
GUIDANCE_RESCALE = 0.7


# VAE decoding
# Mengatur timestep untuk proses decoding VAE
DECODE_TIMESTEP = 0.05

# Mengatur tingkat noise tambahan saat decoding
DECODE_NOISE_SCALE = 0.025

# Mengatur tingkat noise pada gambar referensi
IMAGE_COND_NOISE_SCALE = 0.0


# Reproducibility
# Seed untuk menghasilkan noise secara konsisten
SEED = 42


# Memory optimization
# Mengaktifkan penyimpanan bobot transformer dalam format FP8
USE_FP8_LAYERWISE = True

# Mengaktifkan pemindahan bobot model antara RAM dan VRAM
USE_GROUP_OFFLOAD = True

# Mengaktifkan VAE tiling untuk mengurangi penggunaan VRAM
USE_VAE_TILING = False


# Runtime
# Menggunakan BF16 untuk komputasi model
DTYPE = torch.bfloat16


# Video dimensions
# Menentukan batas maksimum sisi gambar
MAX_SIDE = 832

# Menentukan jumlah frame video
NUM_FRAMES = 75

# Menentukan jumlah frame yang ditampilkan setiap detik
FPS = 24

In [4]:
# Mengizinkan penggunaan TF32 untuk mempercepat operasi matrix multiplication
torch.backends.cuda.matmul.allow_tf32 = True

# Mengizinkan penggunaan TF32 pada operasi convolution
torch.backends.cudnn.allow_tf32 = True

# Mengoptimalkan algoritma convolution berdasarkan ukuran input.
torch.backends.cudnn.benchmark = True

In [5]:
# Memuat transformer dari repositori Hugging Face
transformer = AutoModel.from_pretrained(
  REPO_ID,
  subfolder='transformer',
  dtype=DTYPE,
  low_cpu_mem_usage=True,
)

# Mengaktifkan penyimpanan bobot transformer dalam format FP8
# Komputasi tetap menggunakan BF16 untuk menghemat VRAM
if USE_FP8_LAYERWISE:
  transformer.enable_layerwise_casting(
    storage_dtype=torch.float8_e4m3fn,
    compute_dtype=DTYPE,
  )

# Memuat pipeline LTX menggunakan transformer yang sudah disiapkan
pipeline = LTXConditionPipeline.from_pretrained(
  REPO_ID,
  transformer=transformer,
  dtype=DTYPE,
  low_cpu_mem_usage=True,
)

# Mengaktifkan VAE tiling jika diperlukan untuk menghemat VRAM
if USE_VAE_TILING:
  pipeline.vae.enable_tiling()

# Mengaktifkan group offloading untuk menghemat penggunaan VRAM
if USE_GROUP_OFFLOAD:
  # Menentukan perangkat untuk pemrosesan model
  onloadDevice = torch.device('cuda')
  offloadDevice = torch.device('cpu')
  
  # Memindahkan bagian transformer ke GPU sesuai kebutuhan
  # Leaf-level memungkinkan pemindahan berdasarkan layer terkecil
  pipeline.transformer.enable_group_offload(
    onload_device=onloadDevice,
    offload_device=offloadDevice,
    offload_type='leaf_level',
    use_stream=True,
  )
  
  # Memindahkan text encoder ke GPU dalam kelompok dua blok
  apply_group_offloading(
    pipeline.text_encoder,
    onload_device=onloadDevice,
    offload_type='block_level',
    num_blocks_per_group=2,
  )
  
  # Memindahkan bagian VAE ke GPU sesuai kebutuhan
  apply_group_offloading(
    pipeline.vae,
    onload_device=onloadDevice,
    offload_type='leaf_level',
  )
else:
  # Memuat seluruh pipeline ke GPU jika offloading tidak digunakan
  pipeline.to(device)

d:\miniconda\envs\torch2.15\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [6]:
def prepareImage(imgPath: str, maxSide: int = 832) -> Image.Image:
  # Membuka gambar dan memastikan file ditutup setelah selesai dibaca.
  with Image.open(imgPath) as source:
    # Memperbaiki orientasi berdasarkan metadata EXIF dan mengubah ke RGB.
    img = ImageOps.exif_transpose(source).convert('RGB')
  
  # Mengambil ukuran asli gambar.
  width, height = img.size
  
  # Menghitung skala resize tanpa memperbesar gambar.
  # Sisi terpanjang tidak akan melebihi maxSide.
  scale = min(maxSide / max(width, height), 1.0)
  
  # Menyesuaikan dimensi agar menjadi kelipatan 32.
  # Ukuran minimum setiap sisi adalah 32 piksel.
  width = max(32, round(width * scale / 32) * 32)
  height = max(32, round(height * scale / 32) * 32)
  
  # Mengubah ukuran gambar menggunakan interpolasi LANCZOS.
  img = img.resize(
    (width, height),
    Image.Resampling.LANCZOS,
  )
  
  return img

def generateVideo(
  img: Image.Image,
  prompt: str,
  negativePrompt: str,
) -> list[Image.Image]:
  # Membuat kondisi gambar sebagai frame pertama video.
  condition = LTXVideoCondition(
    video=[img],
    frame_index=0,
  )
  
  # Membuat random generator dengan seed yang sudah ditentukan.
  generator = torch.Generator(device=device).manual_seed(SEED)
  
  # Memastikan seluruh operasi CUDA sebelumnya selesai.
  # Digunakan agar pengukuran waktu lebih akurat.
  torch.cuda.synchronize(device)
  
  # Mengatur ulang pencatatan penggunaan VRAM maksimum.
  torch.cuda.reset_peak_memory_stats(device)
  
  # Memulai pengukuran waktu inference.
  startTime = time.perf_counter()
  
  # Menonaktifkan gradient computation untuk inference.
  with torch.inference_mode():
    # Menjalankan proses generasi video.
    result = pipeline(
      # Menggunakan gambar input sebagai kondisi awal.
      conditions=[condition],

      # Prompt positif dan negatif.
      prompt=prompt,
      negative_prompt=negativePrompt,

      # Resolusi video mengikuti ukuran gambar input.
      width=img.width,
      height=img.height,

      # Jumlah frame yang akan dihasilkan.
      num_frames=NUM_FRAMES,

      # Mengatur jadwal sampling.
      timesteps=TIMESTEPS,

      # Mengatur guidance.
      guidance_scale=GUIDANCE_SCALE,
      guidance_rescale=GUIDANCE_RESCALE,

      # Parameter decoding VAE.
      decode_timestep=DECODE_TIMESTEP,
      decode_noise_scale=DECODE_NOISE_SCALE,
      image_cond_noise_scale=IMAGE_COND_NOISE_SCALE,

      # Menggunakan generator dengan seed yang sudah ditentukan.
      generator=generator,

      # Menghasilkan frame dalam format PIL Image.
      output_type='pil',
    )
  
  # Menunggu seluruh operasi CUDA selesai sebelum menghitung waktu.
  torch.cuda.synchronize()
  
  # Menghitung total waktu inference.
  elapsedTime = time.perf_counter() - startTime
  
  # Mengambil seluruh frame dari hasil pipeline.
  videoFrames = result.frames[0]
  
  # Mengambil penggunaan VRAM maksimum selama inference.
  # Allocated = memori yang digunakan tensor PyTorch.
  peakAllocated = torch.cuda.max_memory_allocated(device) / 1024**3
  # Reserved = memori yang dikelola caching allocator PyTorch.
  peakReserved = torch.cuda.max_memory_reserved(device) / 1024**3
  
  # Menampilkan informasi hasil generasi.
  print(f'Generation time: {elapsedTime:.2f} seconds')
  print(f'Peak allocated VRAM: {peakAllocated:.2f} GB')
  print(f'Peak reserved VRAM: {peakReserved:.2f} GB')
  print(f'Generated frames: {len(videoFrames)}')
  print(f'Resolution: {img.width}x{img.height}')
  print(f'Video duration: {len(videoFrames) / FPS:.2f} seconds')
  
  # Mengembalikan seluruh frame video.
  return videoFrames

In [7]:
# Memuat dan mempersiapkan gambar input.
img = prepareImage(
  f'{pathInputs}/img.jpg',
  MAX_SIDE,
)

# Prompt positif untuk menentukan gerakan karakter.
PROMPT = (
  'Three adorable baby penguins dance happily together. '
  'They smile warmly, gently sway their bodies from side '
  'to side, flap their little wings rhythmically, and '
  'make tiny alternating foot taps. They occasionally '
  'tilt their heads playfully. All three dance in place '
  'with cheerful, synchronized movements. '
  'Preserve their original faces, eyes, beaks, body shapes, '
  'colors, and individual identities. '
  'Keep their original positions and spacing. '
  'Static camera, locked tripod, fixed framing and perspective. '
  'The background and lighting remain completely unchanged.'
)

# Prompt negatif untuk mengurangi gerakan dan perubahan visual yang tidak diinginkan.
NEGATIVE_PROMPT = (
  'camera movement, zoom, pan, tilt, rotation, tracking, '
  'camera shake, reframing, perspective shift, '
  'walking away, moving forward, moving backward, '
  'position changes, jumping, falling, sliding, '
  'face distortion, identity change, morphing, '
  'deformed bodies, extra limbs, duplicated penguins, '
  'background changes, flickering, jittering, blur.'
)

In [8]:
# Menjalankan proses generasi video.
videoFrames = generateVideo(
  img=img,
  prompt=PROMPT,
  negativePrompt=NEGATIVE_PROMPT,
)

# Menggabungkan frame menjadi video MP4.
# FPS menentukan jumlah frame yang diputar setiap detik.
export_to_video(
  videoFrames, 
  f'{pathOutputs}/output.mp4', 
  fps=FPS,
)

`timesteps` were passed without `sigmas`. The timesteps are used as given, while the sigmas derived from them go through the scheduler's shifting, so the two may not be consistent. Pass `sigmas` to get a shifted schedule, or pass both `sigmas` and `timesteps` to control them separately.


  0%|          | 0/8 [00:00<?, ?it/s]

Generation time: 21.89 seconds
Peak allocated VRAM: 8.91 GB
Peak reserved VRAM: 8.93 GB
Generated frames: 73
Resolution: 704x352
Video duration: 3.04 seconds


'..\\res\\outputs/output.mp4'